# Foreign Whispers — End-to-End Pipeline

> **Repository:** This project has its own public GitHub repo at
> [github.com/aegean-ai/foreign-whispers](https://github.com/aegean-ai/foreign-whispers).
> Clone it, file issues, and submit pull requests there.

Commercial dubbing services like [ElevenLabs](https://elevenlabs.io) can take a video, transcribe it, translate it, clone the speaker's voice, and return a dubbed video in the target language — watch their demo below:

<iframe width="560" height="315" src="https://www.youtube.com/embed/RKzp4OfCgBA" frameBorder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowFullScreen></iframe>

You are going to build the same thing from open-source components. No API keys to a proprietary service. No per-minute billing. The entire pipeline runs on your own GPU server.

**Where this matters:**
- **Media localization** — dub documentaries, lectures, or interviews into multiple languages at scale
- **Accessibility** — make video content available to non-English-speaking audiences without manual voiceover
- **Research** — experiment with duration-aware translation, prosody alignment, and speaker-aware TTS in a controllable pipeline
- **Education** — learn how production ML systems compose ASR, MT, TTS, and audio engineering into a single product

This notebook orchestrates the **full pipeline** from YouTube URL to dubbed output video via the `FWClient` SDK. Each step calls the FastAPI backend, which delegates GPU work to the STT and TTS containers.

```
YouTube URL → Download → Transcribe → Translate → TTS (+ alignment) → Stitch → Dubbed Video
```

You will demonstrate your pipeline using the **Foreign Whispers Dubbing Studio** — a Next.js frontend at http://localhost:8501.

![Foreign Whispers Dubbing Studio](images/frontend-dubbing-studio.png)

---

## Architecture

| Layer | What it is | Where it runs |
|-------|-----------|---------------|
| **GPU services** | Whisper STT (port 8000), Chatterbox TTS (port 8020) | Dedicated GPU containers |
| **API** | FastAPI orchestrator (port 8080) — proxies to GPU services | CPU container |
| **`foreign_whispers` library** | Alignment logic, metrics, evaluation | Pure Python — no GPU needed |

```
┌────────────────────┐
│  API (CPU :8080)    │  orchestrates the pipeline
└──┬─────────┬───────┘
   │ HTTP    │ HTTP
   ▼         ▼
┌────────┐ ┌────────┐
│ STT    │ │ TTS    │   GPU containers
│ :8000  │ │ :8020  │
└────────┘ └────────┘
```

## Production tools used

- **[FastAPI](https://fastapi.tiangolo.com/tutorial/first-steps/)**. The backend is a layered FastAPI application with Pydantic schemas, dependency injection, and async request handling.
- **[Logfire](https://www.youtube.com/watch?v=on5RKukQzIg)**. Every pipeline step emits structured traces to Pydantic's observability platform for debugging timing issues and comparing experiment runs.
- **Docker Compose**. The full stack runs as four coordinated containers with GPU passthrough.

## Per-stage integration notebooks

For deep-dives into individual stages, see:

| Notebook | Stage |
| -------- | ----- |
| `download_integration/` | YouTube download + caption fetching |
| `transcription_integration/` | Whisper vs YouTube captions |
| `diarization_integration/` | Speaker diarization (student assignment) |
| `translation_integration/` | argostranslate + duration-aware re-ranking |
| `alignment_integration/` | Temporal alignment: metrics, policies, global optimizer |
| `tts_integration/` | Chatterbox TTS + voice cloning |
| `stitch_integration/` | Final assembly + captions |

## Requirements

```bash
docker compose --profile nvidia up -d   # start GPU services + API
uv sync                                 # install library deps locally
```

### Logfire observability (recommended)

Every API call in this notebook emits a [Logfire](https://logfire.pydantic.dev/) trace span.
To see pipeline execution in the Logfire dashboard, authenticate once:

```bash
uv run logfire auth                     # opens browser for one-time login
```

After authenticating, re-run the notebook — each pipeline stage (P1–P5) will appear as
a span in the Logfire dashboard with timing, video_id, and error details.

If Logfire is not configured, the notebook still runs normally using a no-op shim — no
traces are emitted but nothing breaks.

## Setup — SDK Client and Logfire Tracing

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers import FWClient

API_URL = "http://localhost:8080"
fw = FWClient(API_URL)

# Verify API is reachable
print(fw.healthz())
print(f"SDK client ready: {fw!r}")

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-notebook")
    LOGFIRE_ENABLED = True
    print("Logfire tracing enabled.")
except Exception:
    # Logfire not installed or not authenticated — use no-op shim
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass

    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass

    import types
    logfire = _noop()
    LOGFIRE_ENABLED = False
    print("Logfire not configured — using no-op shim. Run `logfire auth` to enable.")

Project root: /root/foreign-whispers
{'status': 'ok'}
SDK client ready: FWClient('http://localhost:8080')


Logfire not configured — using no-op shim. Run `logfire auth` to enable.


---
## Pipeline Execution

Each step calls the API via the `FWClient` SDK. All GPU work happens in the STT/TTS containers. Results are cached on disk — re-running skips already-completed steps.

### P1 — Download

In [2]:
VIDEO_URL = "https://www.youtube.com/watch?v=GYQ5yGV_-Oc"

with logfire.span("P1.download"):
    dl = fw.download(VIDEO_URL)

video_id = dl["video_id"]
print(f"Video ID : {video_id}")
print(f"Title    : {dl['title']}")
print(f"Captions : {len(dl['caption_segments'])} segments")
for seg in dl["caption_segments"][:5]:
    print(f"  {seg}")

Video ID : GYQ5yGV_-Oc
Title    : Strait of Hormuz disruption threatens to shake global economy
Captions : 170 segments
  {'start': 2.32, 'end': None, 'text': '60 Minutes overtime.', 'duration': 3.8}
  {'start': 6.48, 'end': None, 'text': "What's the worst case scenario that", 'duration': 3.76}
  {'start': 8.0, 'end': None, 'text': "you're worried about is that it is", 'duration': 4.4}
  {'start': 10.24, 'end': None, 'text': 'closed for weeks and weeks and weeks', 'duration': 5.519}
  {'start': 12.4, 'end': None, 'text': 'here and you start to see the global', 'duration': 5.84}


### P2 — Transcribe

In [3]:
with logfire.span("P2.transcribe", video_id=video_id):
    transcript = fw.transcribe(video_id)

print(f"Language : {transcript['language']}")
print(f"Segments : {len(transcript['segments'])}")
print(f"Skipped  : {transcript.get('skipped', False)}")
print("\nFirst 3 segments:")
for seg in transcript["segments"][:3]:
    dur = seg["end"] - seg["start"]
    print(f"  [{seg['start']:.1f}s – {seg['end']:.1f}s ({dur:.1f}s)]  {seg['text'].strip()}")

Language : en
Segments : 170
Skipped  : True

First 3 segments:
  [2.3s – 6.1s (3.8s)]  60 Minutes overtime.
  [6.5s – 10.2s (3.8s)]  What's the worst case scenario that
  [8.0s – 12.4s (4.4s)]  you're worried about is that it is


### P3 — Translate

In [4]:
with logfire.span("P3.translate", video_id=video_id):
    translation = fw.translate(video_id)

print(f"Target language: {translation['target_language']}")
print(f"Segments:        {len(translation['segments'])}")
print("\nFirst segment comparison:")
en_seg = transcript["segments"][0]
es_seg = translation["segments"][0]
print(f"  EN: {en_seg['text']}")
print(f"  ES: {es_seg['text']}")

Target language: es
Segments:        170

First segment comparison:
  EN: 60 Minutes overtime.
  ES: 60 minutos horas extra.


### P4 — TTS

In [5]:
FINAL_CONFIG = "c-e5604bc"

with logfire.span("P4.tts_cached", video_id=video_id, config=FINAL_CONFIG):
    tts_result = fw.tts(video_id, config=FINAL_CONFIG, alignment=False)

print(f"Audio path: {tts_result['audio_path']}")
print(f"Config:     {tts_result.get('config', 'N/A')}")

Audio path: /app/pipeline_data/api/tts_audio/chatterbox/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.wav
Config:     c-e5604bc


### P5 — Stitch

In [6]:
with logfire.span("P5.stitch", video_id=video_id, config=FINAL_CONFIG):
    stitch_result = fw.stitch(video_id, config=FINAL_CONFIG)

print(f"Video path: {stitch_result['video_path']}")
print(f"Config:     {stitch_result.get('config', FINAL_CONFIG)}")

Video path: /app/pipeline_data/api/dubbed_videos/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.mp4
Config:     c-e5604bc


## Summary

| Step | Tool | Output |
|------|------|--------|
| P1 — Download | `yt-dlp` via API | `videos/*.mp4`, `youtube_captions/*.txt` |
| P2 — Transcribe | Whisper STT (GPU) | `transcriptions/whisper/*.json` |
| P3 — Translate | `argostranslate` | `translations/argos/*.json` |
| P4 — TTS | Chatterbox (GPU) | `tts_audio/chatterbox/{config}/*.wav` |
| P5 — Stitch | `ffmpeg` | `dubbed_videos/{config}/*.mp4`, `dubbed_captions/*.vtt` |

All artifacts are cached in `pipeline_data/api/`. Re-running skips completed steps.

For alignment analysis, evaluation, and per-stage deep-dives, see the integration notebooks listed in the intro.

In [7]:
import json

# Show pipeline artifacts
data_dir = PROJECT_ROOT / "pipeline_data" / "api"
TITLE = "Strait of Hormuz disruption threatens to shake global economy"
FINAL_CONFIG = "c-e5604bc"

final_artifacts = {
    "Source video": data_dir / "videos" / f"{TITLE}.mp4",
    "YouTube captions": data_dir / "youtube_captions" / f"{TITLE}.txt",
    "Transcription": data_dir / "transcriptions" / "whisper" / f"{TITLE}.json",
    "Translation": data_dir / "translations" / "argos" / f"{TITLE}.json",
    "TTS audio": data_dir / "tts_audio" / "chatterbox" / FINAL_CONFIG / f"{TITLE}.wav",
    "Dubbed video": data_dir / "dubbed_videos" / FINAL_CONFIG / f"{TITLE}.mp4",
    "Dubbed captions": data_dir / "dubbed_captions" / f"{TITLE}.vtt",
}

print("=== Final Pipeline Artifacts ===")
for label, path in final_artifacts.items():
    print(f"{label:<18} exists={path.exists()}  {path}")
    if path.exists() and path.is_file():
        print(f"{'':<18} size={path.stat().st_size / (1024 * 1024):.2f} MB")

# Show first few lines of the final dubbed captions (TTS-scheduled WebVTT format)
final_vtt = final_artifacts["Dubbed captions"]

if final_vtt.exists():
    print(f"\n=== Dubbed Captions Preview ({final_vtt.name}) ===")
    for line in final_vtt.read_text().splitlines()[:20]:
        print(f"  {line}")
else:
    print("\nFinal dubbed captions not found.")

=== Final Pipeline Artifacts ===
Source video       exists=True  /root/foreign-whispers/pipeline_data/api/videos/Strait of Hormuz disruption threatens to shake global economy.mp4
                   size=34.31 MB
YouTube captions   exists=True  /root/foreign-whispers/pipeline_data/api/youtube_captions/Strait of Hormuz disruption threatens to shake global economy.txt
                   size=0.01 MB
Transcription      exists=True  /root/foreign-whispers/pipeline_data/api/transcriptions/whisper/Strait of Hormuz disruption threatens to shake global economy.json
                   size=0.03 MB
Translation        exists=True  /root/foreign-whispers/pipeline_data/api/translations/argos/Strait of Hormuz disruption threatens to shake global economy.json
                   size=0.03 MB
TTS audio          exists=True  /root/foreign-whispers/pipeline_data/api/tts_audio/chatterbox/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.wav
                   size=18.58 MB
Dubbed vide